## Shared Working Directory (run this first, in every notebook)
Each Colab **tab** runs in its own isolated virtual machine, so a plain
`results/outputs/...` path only exists inside that one tab. To let all
seven notebooks (six individual + `group_pipeline.ipynb`) read and write
the *same* `results/` folder — which `group_pipeline.ipynb` Part C needs
— mount Google Drive and switch into one shared project folder there.

**Before running:** make sure a folder named `2026-Y2-S1-MLB-B7G1-08` (with this
project's contents) exists directly under **My Drive**. If it's nested
inside another folder, edit `PROJECT_DIR` below to match the real path.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/2026-Y2-S1-MLB-B7G1-08"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
except ImportError:
    pass  # not running in Colab (e.g. local Jupyter) -- uses the current directory instead

print("Working directory:", os.getcwd())


# Classification: Logistic Regression
**Member:** Ishki M A  **IT Number:** IT25103555
**Assigned model category:** Classification

This notebook fulfils the implementation details, tuning method, number
and variety of models trained, evaluation metrics per variety, comparison
across varieties with a conclusion, and limitations/improvements.

**Preprocessing note:** to avoid the data-leakage issue found in Progress
Review I, the split happens first, and the imputer/scaler are fit only on
`X_train` then applied to `X_test` — never fit on test data.

## 1. Model Suitability
Logistic Regression is a strong baseline for this dataset: it's a linear
model that outputs a probability, which suits a binary medical screening
task (diabetic vs not), is fast to train on ~614 samples, and its
coefficients are directly interpretable — useful for explaining which
clinical features drive risk. It assumes a roughly linear relationship
between features and log-odds of the outcome, which is a reasonable starting
assumption here before trying more flexible models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"
df = pd.read_csv("data/raw/diabetes.csv") if os.path.exists("data/raw/diabetes.csv") else pd.read_csv(url)

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

zero_as_missing_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
X = df.drop(columns="Outcome").copy()
y = df["Outcome"]
X[zero_as_missing_cols] = X[zero_as_missing_cols].replace(0, np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

imputer = SimpleImputer(strategy="median")
X_train_imp = X_train.copy()
X_test_imp = X_test.copy()
X_train_imp[zero_as_missing_cols] = imputer.fit_transform(X_train[zero_as_missing_cols])
X_test_imp[zero_as_missing_cols] = imputer.transform(X_test[zero_as_missing_cols])

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train_imp.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test_imp.columns)

print("Train:", X_train_scaled.shape, " Test:", X_test_scaled.shape)


## 2. Implementation Details
Using `sklearn.linear_model.LogisticRegression`. Key hyperparameters explored:
- `C` — inverse regularization strength (lower C = stronger regularization)
- `class_weight` — `None` vs `"balanced"`, to address the moderate class
  imbalance (500 non-diabetic vs 268 diabetic)
- `solver` — default `lbfgs` throughout, suitable for this dataset size

## 3. Training Multiple Varieties (Manual + GridSearchCV Tuning)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

varieties = {
    "Default (C=1, no weighting)": LogisticRegression(C=1.0, max_iter=2000),
    "Strong regularization (C=0.1)": LogisticRegression(C=0.1, max_iter=2000),
    "Weak regularization (C=10)": LogisticRegression(C=10.0, max_iter=2000),
    "Balanced class weight (C=1)": LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000),
}

variety_results = []
roc_store = {}
for name, model in varieties.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    proba = model.predict_proba(X_test_scaled)[:, 1]
    roc_store[name] = proba
    variety_results.append({
        "Variety": name,
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "Precision": round(precision_score(y_test, preds), 4),
        "Recall": round(recall_score(y_test, preds), 4),
        "F1": round(f1_score(y_test, preds), 4),
        "ROC-AUC": round(roc_auc_score(y_test, proba), 4),
    })

results_df = pd.DataFrame(variety_results)
results_df


## 4. GridSearchCV — Systematic Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.01, 0.1, 0.5, 1, 2, 5, 10],
    "class_weight": [None, "balanced"],
}
grid = GridSearchCV(
    LogisticRegression(max_iter=2000), param_grid, cv=5, scoring="f1", n_jobs=-1
)
grid.fit(X_train_scaled, y_train)
print("Best params:", grid.best_params_)
print("Best CV F1:", round(grid.best_score_, 4))

best_model = grid.best_estimator_
preds = best_model.predict(X_test_scaled)
proba = best_model.predict_proba(X_test_scaled)[:, 1]
print("\nTest Accuracy:", round(accuracy_score(y_test, preds), 4))
print("Test Recall (Diabetes):", round(recall_score(y_test, preds), 4))
print("Test F1 (Diabetes):", round(f1_score(y_test, preds), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, proba), 4))

roc_store["Tuned (GridSearchCV)"] = proba


## EDA Visualization: Comparing All Varieties

In [ ]:
plt.figure(figsize=(9, 5))
metrics_to_plot = results_df.set_index("Variety")[["Accuracy", "Recall", "F1"]]
metrics_to_plot.plot(kind="bar", figsize=(9, 5))
plt.title("Logistic Regression — Metric Comparison Across Varieties")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
os.makedirs("results/eda_visualizations", exist_ok=True)
plt.savefig("results/eda_visualizations/lr_variety_comparison.png", dpi=150)
plt.show()


## EDA Visualization: ROC Curves Across Varieties

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(7, 6))
for name, proba in roc_store.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance level")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Logistic Regression \u2014 ROC Curves Across Varieties")
plt.legend(fontsize=8)
plt.tight_layout()
os.makedirs("results/eda_visualizations", exist_ok=True)
plt.savefig("results/eda_visualizations/lr_roc_curves.png", dpi=150)
plt.show()


## EDA Visualization: Confusion Matrix (Tuned Model)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, best_model.predict(X_test_scaled))
plt.figure(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Diabetes", "Diabetes"],
            yticklabels=["No Diabetes", "Diabetes"])
plt.title("Logistic Regression \u2014 Confusion Matrix (Tuned Model)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
os.makedirs("results/eda_visualizations", exist_ok=True)
plt.savefig("results/eda_visualizations/lr_confusion_matrix.png", dpi=150)
plt.show()


## 4b. Preprocessing Varieties from Progress Review I + 5-Fold Stratified Cross-Validation
The varieties above only change **hyperparameters**. The brief also asks for varieties with
**different pre-processing**, so this section re-uses the Progress Review I techniques of every
member as switchable steps and measures their effect on this model:

| Variety | Steps applied (all fitted on the training fold only) |
|---|---|
| A | Missing-data handling (median imputation) — Ishki M A |
| B | A + StandardScaler — Jayaratne L S |
| C | A + MinMaxScaler — Jayaratne L S |
| D | B + IQR outlier removal on BloodPressure/BMI (training rows only) — Ligasaran N |
| E | B + engineered BMI category / Age group encoding — Pathirana M P K S J |
| F | B + SelectKBest (ANOVA F-test, k=5) — Navoda W P T |

**Validation method:** 5-fold Stratified Cross-Validation on the 614 training rows only.
Every step is re-fitted inside each fold, so no information leaks from the validation fold,
and the 154-row test set stays untouched until the final check. Mean scores over 5 folds are
more reliable than one 80/20 split (a single split of 154 rows can move F1 by several points).

In [ ]:
# ---- Shared helper: Progress Review I preprocessing steps, fitted per CV fold ----
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Each variety switches Progress Review I techniques on/off.
# Missing-data handling (median imputation) is always on: zeros are not valid values.
PREPROCESSING_VARIETIES = {
    "A. Imputation only":            dict(scaler=None,       outliers=False, encode=False, select_k=None),
    "B. A + StandardScaler":         dict(scaler="standard", outliers=False, encode=False, select_k=None),
    "C. A + MinMaxScaler":           dict(scaler="minmax",   outliers=False, encode=False, select_k=None),
    "D. B + IQR outlier removal":    dict(scaler="standard", outliers=True,  encode=False, select_k=None),
    "E. B + BMI/Age encoding":       dict(scaler="standard", outliers=False, encode=True,  select_k=None),
    "F. B + SelectKBest (k=5)":      dict(scaler="standard", outliers=False, encode=False, select_k=5),
}

def _encode(X):
    # Pathirana's encoding step: BMI category (one-hot) + ordinal age group
    X = X.copy()
    bmi_cat = pd.cut(X["BMI"], bins=[0, 18.5, 25, 30, np.inf],
                     labels=["Underweight", "Normal", "Overweight", "Obese"], right=False)
    X = pd.concat([X, pd.get_dummies(bmi_cat, prefix="BMI").astype(int)], axis=1)
    X["Age_Group_Encoded"] = pd.cut(X["Age"], bins=[0, 30, 50, np.inf], labels=[0, 1, 2], right=False).astype(int)
    return X

def preprocess_fold(X_tr, y_tr, X_va, scaler=None, outliers=False, encode=False, select_k=None):
    """Fit every step on the training fold only, then apply it to the validation fold."""
    # 1. Missing data (Ishki M A): median imputation
    imp = SimpleImputer(strategy="median").fit(X_tr)
    X_tr = pd.DataFrame(imp.transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_va = pd.DataFrame(imp.transform(X_va), columns=X_va.columns, index=X_va.index)
    # 2. Outlier removal (Ligasaran N): IQR, k=1.5, on BloodPressure & BMI -- training rows only
    if outliers:
        keep = pd.Series(True, index=X_tr.index)
        for col in ["BloodPressure", "BMI"]:
            q1, q3 = X_tr[col].quantile([0.25, 0.75])
            iqr = q3 - q1
            keep &= X_tr[col].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
        X_tr, y_tr = X_tr[keep], y_tr[keep]
    # 3. Encoding (Pathirana M P K S J): engineered BMI / Age categories
    if encode:
        X_tr, X_va = _encode(X_tr), _encode(X_va)
        X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)
    # 4. Feature selection (Navoda W P T): ANOVA F-test
    if select_k:
        sel = SelectKBest(f_classif, k=select_k).fit(X_tr, y_tr)
        cols = X_tr.columns[sel.get_support()]
        X_tr, X_va = X_tr[cols], X_va[cols]
    # 5. Scaling (Jayaratne L S)
    if scaler is not None:
        sc = (StandardScaler() if scaler == "standard" else MinMaxScaler()).fit(X_tr)
        X_tr, X_va = sc.transform(X_tr), sc.transform(X_va)
    return np.asarray(X_tr, dtype=float), y_tr, np.asarray(X_va, dtype=float)

def _scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

def cv_preprocessing_comparison(model, X_raw, y, varieties=PREPROCESSING_VARIETIES, n_splits=5):
    """5-fold Stratified CV on the TRAINING set only; returns mean and std per metric."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    rows = []
    for name, opts in varieties.items():
        fold = {"Accuracy": [], "Precision": [], "Recall": [], "F1": [], "ROC-AUC": []}
        for tr_idx, va_idx in skf.split(X_raw, y):
            Xtr, ytr, Xva = preprocess_fold(X_raw.iloc[tr_idx], y.iloc[tr_idx], X_raw.iloc[va_idx], **opts)
            yva = y.iloc[va_idx]
            m = clone(model).fit(Xtr, ytr)
            p = m.predict(Xva)
            fold["Accuracy"].append(accuracy_score(yva, p))
            fold["Precision"].append(precision_score(yva, p, zero_division=0))
            fold["Recall"].append(recall_score(yva, p))
            fold["F1"].append(f1_score(yva, p))
            fold["ROC-AUC"].append(roc_auc_score(yva, _scores(m, Xva)))
        row = {"Preprocessing Variety": name}
        for k, v in fold.items():
            row[f"CV {k}"] = round(np.mean(v), 4)
        row["CV F1 Std"] = round(np.std(fold["F1"]), 4)
        rows.append(row)
    return pd.DataFrame(rows)

def test_best_variety(model, cv_df, X_train_raw, y_train, X_test_raw, y_test, metric="CV F1"):
    """Refit the best CV variety on the full training set and check it once on the held-out test set."""
    best = cv_df.loc[cv_df[metric].idxmax(), "Preprocessing Variety"]
    Xtr, ytr, Xte = preprocess_fold(X_train_raw, y_train, X_test_raw, **PREPROCESSING_VARIETIES[best])
    m = clone(model).fit(Xtr, ytr)
    p = m.predict(Xte)
    res = {"Best Preprocessing": best,
           "Test Accuracy": round(accuracy_score(y_test, p), 4),
           "Test Precision": round(precision_score(y_test, p, zero_division=0), 4),
           "Test Recall": round(recall_score(y_test, p), 4),
           "Test F1": round(f1_score(y_test, p), 4),
           "Test ROC-AUC": round(roc_auc_score(y_test, _scores(m, Xte)), 4)}
    return res, m

In [ ]:
# Hyperparameters = the GridSearchCV winner from Section 4 (C=0.1, balanced)
cv_model = LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)

cv_df = cv_preprocessing_comparison(cv_model, X_train, y_train)
os.makedirs("results/outputs", exist_ok=True)
cv_df.to_csv("results/outputs/cv_preprocessing_lr.csv", index=False)
cv_df

## EDA Visualization: Effect of Each Preprocessing Variety (5-Fold CV)

In [ ]:
ax = cv_df.set_index("Preprocessing Variety")[["CV Accuracy", "CV Recall", "CV F1", "CV ROC-AUC"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1))
ax.set_title("Logistic Regression \u2014 Preprocessing Varieties (5-Fold Stratified CV)")
ax.set_ylabel("Mean score over 5 folds")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
os.makedirs("results/eda_visualizations", exist_ok=True)
plt.savefig("results/eda_visualizations/lr_preprocessing_cv.png", dpi=150)
plt.show()

### Held-out test check of the best preprocessing variety
The variety with the highest mean CV F1 is refitted on the full training set and evaluated **once** on the untouched test set.

In [ ]:
best_prep_result, best_prep_model = test_best_variety(cv_model, cv_df, X_train, y_train, X_test, y_test)
best_prep_result["Model"] = "Logistic Regression"
best_prep_result["Member"] = "Ishki M A (IT25103555)"
best_prep_result["Best CV F1"] = float(cv_df["CV F1"].max())
pd.DataFrame([best_prep_result]).to_csv("results/outputs/best_preprocessing_lr.csv", index=False)
best_prep_result

### Findings — preprocessing varieties (5-fold CV)
- **Scaling helps a little:** StandardScaler (B) raised CV F1 from 0.665 (A, unscaled) to 0.682. Regularisation
  (C) penalises all coefficients equally, so features need comparable ranges.
- **Encoding helped most:** adding the engineered BMI-category/Age-group features (E) gave the best CV F1
  (**0.697**) and recall (0.762). These categories let a linear model express simple non-linear thresholds
  (e.g. "obese" vs "normal") that raw BMI cannot.
- **Outlier removal (D) and feature selection (F) did not help** (CV F1 0.666 and 0.664): removing rows
  shrinks an already small training set, and dropping 3 features loses a little signal.
- **Held-out test:** the best variety (E) scored Accuracy 0.753, Recall 0.778, F1 **0.689**, ROC-AUC 0.826 —
  better than every hyperparameter-only variety above (best F1 0.650).

## Save Best Result (for the group cross-member comparison)

In [ ]:
best_row = results_df.loc[results_df["F1"].idxmax()].to_dict()
best_row["Model"] = "Logistic Regression"
best_row["Member"] = "Ishki M A (IT25103555)"
best_row["Tuned Accuracy"] = round(accuracy_score(y_test, best_model.predict(X_test_scaled)), 4)
best_row["Tuned F1"] = round(f1_score(y_test, best_model.predict(X_test_scaled)), 4)
best_row["Tuned ROC-AUC"] = round(roc_auc_score(y_test, best_model.predict_proba(X_test_scaled)[:, 1]), 4)

os.makedirs("../results/outputs", exist_ok=True)
os.makedirs("results/outputs", exist_ok=True)
pd.DataFrame([best_row]).to_csv("results/outputs/best_result_logistic_regression.csv", index=False)
best_row


## 5. Comparison & Conclusion
On the single 80/20 split, the **balanced class-weight** variety (C=1) gave the best F1 (0.650) and recall
(0.704), versus F1 0.546 for the unweighted default — class weighting matters more than the value of C.
GridSearchCV (5-fold, F1-scored) chose C=0.1 + balanced (CV F1 0.676), which scored F1 0.615 on the test
split; the gap between the two shows how much a 154-row test set can vary.

The preprocessing study (Section 4b) was more decisive: **StandardScaler + BMI/Age encoding (variety E)**
gave the best 5-fold CV F1 (0.697) and the best held-out result (F1 0.689, recall 0.778, ROC-AUC 0.826).
**Final choice: Logistic Regression, C=0.1, balanced weights, scaled + encoded features.**

## 6. Limitations & Possible Improvements
Logistic Regression assumes a linear relationship with the log-odds; the encoding step only partly works
around this. Interaction terms (e.g. Glucose × BMI), KNN/iterative imputation instead of the median, and
threshold tuning (choosing a probability cut-off that targets a set recall) are natural next steps.